# Limpeza e Preparação dos Dados

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# adiciona a raiz do projeto ao caminho de busca do Python
sys.path.append(str(Path().resolve().parent))

## 1. Criando cópias dos conjuntos de dados para serem manipulados

In [2]:
df_train = pd.read_csv('../data/raw/train.csv.zip')
df_test = pd.read_csv('../data/raw/test.csv.zip')

/tmp/ipykernel_4916/3186401841.py:1: DtypeWarning: Columns (0: Monthly_Balance) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('../data/raw/train.csv.zip')


In [3]:
df_train_copy = df_train.copy()
df_test_copy = df_test.copy()

In [4]:
# forçando pandas a mostrar todo o conteúdo das células
pd.set_option('display.max_colwidth', None)

problemas = pd.read_csv('../data/external/achados_etapa_1.csv')
problemas

,Problema identificado,Variáveis,Ação prevista
0,Valores numéricos em formato de string,"Age, Annual_Income, Num_of_Loan, Num_of_Delayed_Payment, Changed_Credit_Limit, Outstanding_Debt, Amount_invested_monthly, Monthly_Balance",Corrigir valores no formato incorreto e converter para número
1,Valores discretos em formato contínuo,Num_Credit_Inquiries,Converter para int
2,Variáveis que devem ter seus valores transformados,"Credit_History_Age, Payment_of_Min_Amount",Ver qual a melhor abordagem para cada caso
3,Valores inconsistentes com o domínio do problema,"Num_Bank_Accounts, Delay_from_due_date","Investigar e corrigir, ou substituir por valores ausentes"
4,Ruídos em variáveis categóricas,Verificar todas as variáveis,Investigar e substituir por valores ausentes
5,Variável alvo com instâncias desbalanceadas,Credit_Score,"Verificar necessidade de técnicas para balanceamento durante a modelagem, ou validação cruzada"
6,Variáveis identificadoras para serem excluídas,"ID, Customer_ID, Name, SSN",Excluir colunas na etapa de limpeza
7,Valores ausentes,"Monthly_Inhand_Salary, Type_of_Loan, Name, Credit_History_Age, Num_of_Delayed_Payment, Amount_invested_monthly, Num_Credit_Inquiries, Monthly_Balance",Definir estratégia de imputação


>Após a etapa 1 da pipeline, encontramos alguns problemas que precisamos tratar na limpeza de dados, conforme mostrado na tabela acima.

In [5]:
dicionario = pd.read_csv('../data/external/dictionary.csv')

## 2. Tratando valores numéricos em formato de string

Para tratar esses valores, precisamos saber quais são os formatos "sujos" de números presentes nas colunas. Por isso, construímos funções para encontrar os ruídos nos dados que deveriam ser int e nos que deveriam ser float. Após encontrar quais são os caracteres que estão transformando os números em strings, fazemos uma função para remover essas sujeiras. As entradas que não contém números virarão dados nulos.

### 2.1 Tratando valores que deveriam ser int no dataset de treino

In [6]:
import src.funcoes_limpeza.limpeza_int as limpeza_int

# pegar as variáveis que deveriam ser int que tem o problema de estarem em formato de string
variaveis_inteiras = limpeza_int.encontrar_variaveis_int(problemas, dicionario)
print(f"Valores em formato de string que deveriam ser int: {variaveis_inteiras}")

Valores em formato de string que deveriam ser int: ['Age', 'Num_of_Loan', 'Num_of_Delayed_Payment']


In [7]:
# procurar os ruídos que essas variáveis possuem
for indice, variaveis in enumerate(variaveis_inteiras):
    limpeza_int.encontrar_ruidos_int(df_train_copy, variaveis_inteiras[indice])


Coluna: Age
Valores distintos com ruído: 128
Total de registros com ruído: 5825

Ocorrências:
Age
-500     886
24_      161
38_      161
29_      160
26_      153
        ... 
6921_      1
780_       1
1070_      1
5798_      1
4808_      1
Name: count, Length: 128, dtype: int64

Coluna: Num_of_Loan
Valores distintos com ruído: 36
Total de registros com ruído: 8661

Ocorrências:
Num_of_Loan
-100     3876
2_        782
4_        727
3_        718
0_        550
1_        523
7_        414
6_        398
5_        332
9_        160
8_        156
597_        1
92_         1
1347_       1
1185_       1
235_        1
1459_       1
1320_       1
630_        1
359_        1
1225_       1
131_        1
1311_       1
1129_       1
785_        1
143_        1
1131_       1
27_         1
1171_       1
227_        1
378_        1
1219_       1
527_        1
1027_       1
696_        1
1132_       1
Name: count, dtype: int64

Coluna: Num_of_Delayed_Payment
Valores distintos com ruído: 54
Total de re

---
Vemos que as sujeiras nos dados consistem em caracteres "-" e "_" no início e no final dos números. Faremos uma função para pegar esses caracteres e excluí-los dos números. Os números que estão em formato negativo deverão virar nulos, pois não fazem sentido no contexto do problema.

---

In [8]:
# limpando os valores com ruído
for indice, variaveis in enumerate(variaveis_inteiras):
    limpeza_int.limpar_coluna_int(df_train_copy, variaveis)


Coluna: Age
Valores com '_' corrigidos: 4939
Valores negativos convertidos para NaN: 886

Coluna: Num_of_Loan
Valores com '_' corrigidos: 4785
Valores negativos convertidos para NaN: 3876

Coluna: Num_of_Delayed_Payment
Valores com '_' corrigidos: 2744
Valores negativos convertidos para NaN: 644


In [9]:
# verificando se ainda restam ruídos
for indice, variaveis in enumerate(variaveis_inteiras):
    limpeza_int.encontrar_ruidos_int(df_train_copy, variaveis_inteiras[indice])


Coluna: Age
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Num_of_Loan
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Num_of_Delayed_Payment
Valores distintos com ruído: 0
Total de registros com ruído: 0


In [10]:
# verificando os tipos de dados das colunas após a limpeza
for variaveis in variaveis_inteiras:
    print(f'{variaveis}: {df_train_copy[variaveis].dtype}')

Age: Int64
Num_of_Loan: Int64
Num_of_Delayed_Payment: Int64


> As variáveis do dataset de treino foram limpas e convertidas para int. Precisamos fazer o mesmo com o dataset de teste.

### 2.2 Tratando valores que deveriam ser int no dataset de teste

In [11]:
# procurando ruídos no dataset de teste
for indice, variaveis in enumerate(variaveis_inteiras):
    limpeza_int.encontrar_ruidos_int(df_test_copy, variaveis_inteiras[indice])


Coluna: Age
Valores distintos com ruído: 86
Total de registros com ruído: 2941

Ocorrências:
Age
-500     464
32_       89
36_       82
19_       78
39_       77
        ... 
831_       1
217_       1
8340_      1
7868_      1
1401_      1
Name: count, Length: 86, dtype: int64

Coluna: Num_of_Loan
Valores distintos com ruído: 19
Total de registros com ruído: 4410

Ocorrências:
Num_of_Loan
-100     1974
3_        400
4_        386
2_        342
0_        283
1_        266
7_        197
6_        195
5_        180
9_         91
8_         88
481_        1
954_        1
725_        1
72_         1
734_        1
519_        1
1292_       1
1296_       1
Name: count, dtype: int64

Coluna: Num_of_Delayed_Payment
Valores distintos com ruído: 48
Total de registros com ruído: 1702

Ocorrências:
Num_of_Delayed_Payment
-1       123
-2       103
20_       96
17_       91
16_       89
19_       85
15_       80
12_       80
8_        78
9_        75
10_       74
11_       64
18_       61
13_       

In [12]:
# limpando os valores com ruído
for indice, variaveis in enumerate(variaveis_inteiras):
    limpeza_int.limpar_coluna_int(df_test_copy, variaveis)


Coluna: Age
Valores com '_' corrigidos: 2477
Valores negativos convertidos para NaN: 464

Coluna: Num_of_Loan
Valores com '_' corrigidos: 2436
Valores negativos convertidos para NaN: 1974

Coluna: Num_of_Delayed_Payment
Valores com '_' corrigidos: 1427
Valores negativos convertidos para NaN: 287


In [13]:
# verificando se ainda restam ruídos
for indice, variaveis in enumerate(variaveis_inteiras):
    limpeza_int.encontrar_ruidos_int(df_test_copy, variaveis_inteiras[indice])


Coluna: Age
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Num_of_Loan
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Num_of_Delayed_Payment
Valores distintos com ruído: 0
Total de registros com ruído: 0


In [14]:
# verificando os tipos de dados das colunas após a limpeza
for variaveis in variaveis_inteiras:
    print(f'{variaveis}: {df_test_copy[variaveis].dtype}')

Age: Int64
Num_of_Loan: Int64
Num_of_Delayed_Payment: Int64


---
O problema de "Valores numéricos em formato de string" foi resolvido para dados inteiros em ambos os datasets. Agora iremos resolver para dados que deveriam ser float.

---

### 2.3 Tratando valores que deveriam ser float no dataset de treino

In [15]:
import src.funcoes_limpeza.limpeza_float as limpeza_float

# pegar as variáveis que deveriam ser int que tem o problema de estarem em formato de string
variaveis_float = limpeza_float.encontrar_variaveis_float(problemas, dicionario)
print(f"Valores em formato de string que deveriam ser float: {variaveis_float}")

Valores em formato de string que deveriam ser float: ['Annual_Income', 'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly', 'Monthly_Balance']


In [16]:
# procurar os ruídos que essas variáveis possuem
for indice, variaveis in enumerate(variaveis_float):
    limpeza_float.encontrar_ruidos_float(df_train_copy, variaveis_float[indice])


Coluna: Annual_Income
Valores distintos com ruído: 5503
Total de registros com ruído: 6980

Ocorrências:
Annual_Income
14388.79_    5
42904.26_    5
9275.95_     4
10024.25_    4
33381.68_    4
            ..
64511.34_    1
41329.56_    1
38321.39_    1
16680.35_    1
37188.1_     1
Name: count, Length: 5503, dtype: int64

Coluna: Changed_Credit_Limit
Valores distintos com ruído: 1
Total de registros com ruído: 2091

Ocorrências:
Changed_Credit_Limit
_    2091
Name: count, dtype: int64

Coluna: Outstanding_Debt
Valores distintos com ruído: 975
Total de registros com ruído: 1009

Ocorrências:
Outstanding_Debt
396.97_     3
1054.86_    2
761.18_     2
1172.87_    2
289.59_     2
           ..
494.51_     1
807.65_     1
1095.9_     1
1350.85_    1
1453.61_    1
Name: count, Length: 975, dtype: int64

Coluna: Amount_invested_monthly
Valores distintos com ruído: 1
Total de registros com ruído: 4305

Ocorrências:
Amount_invested_monthly
__10000__    4305
Name: count, dtype: int64

Coluna: 

---

As sujeiras nos valores são o caractere "_" , que aparece sozinho (em `Changed_Credit_Limit`) ou no início ou no final do número. Nas variáveis `Amount_invested_monthly` e `Monthly_Balance`, o caractere aparece duas vezes no início e no final dos números. Faremos uma função para remover o caractere dos números e convertê-los para float. Quando o caractere aparecer sozinho, o valor deve virar nulo, pois não há número junto dele para ser aproveitado.

---

In [17]:
# limpando os valores com ruído
for indice, variaveis in enumerate(variaveis_float):
    limpeza_float.limpar_coluna_float(df_train_copy, variaveis)


Coluna: Annual_Income
Valores com '_' corrigidos: 6980
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0

Coluna: Changed_Credit_Limit
Valores com '_' corrigidos: 2091
Valores compostos apenas por '_': 2091
Valores convertidos para NaN: 2091

Coluna: Outstanding_Debt
Valores com '_' corrigidos: 1009
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0

Coluna: Amount_invested_monthly
Valores com '_' corrigidos: 4305
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0

Coluna: Monthly_Balance
Valores com '_' corrigidos: 9
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0


In [18]:
# verificando se ainda restam ruídos
for indice, variaveis in enumerate(variaveis_float):
    limpeza_float.encontrar_ruidos_float(df_train_copy, variaveis_float[indice])


Coluna: Annual_Income
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Changed_Credit_Limit
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Outstanding_Debt
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Amount_invested_monthly
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Monthly_Balance
Valores distintos com ruído: 1
Total de registros com ruído: 9

Ocorrências:
Monthly_Balance
-3.333333333333333e+26    9
Name: count, dtype: int64


In [19]:
# verificando os tipos de dados das colunas após a limpeza
for variaveis in variaveis_float:
    print(f'{variaveis}: {df_train_copy[variaveis].dtype}')

Annual_Income: float64
Changed_Credit_Limit: float64
Outstanding_Debt: float64
Amount_invested_monthly: float64
Monthly_Balance: float64


---
A função para encontrar ruídos aponta o número "-333333333333333333333333333" como sujeira, mas isso ocorre somente porque o número é muito grande e é convertido para notação científica. Após isso, vemos que a coluna `Monthly_Balance`, que possui este número, está com o tipo float. Então a limpeza foi concluída, porém o valor apontado deve ser tratado posteriomente como outlier.

---

> As variáveis do dataset de treino foram limpas e convertidas para float. Precisamos fazer o mesmo com o dataset de teste.

### 2.4 Tratando valores que deveriam ser float no dataset de teste

In [20]:
# procurar os ruídos que essas variáveis possuem
for indice, variaveis in enumerate(variaveis_float):
    limpeza_float.encontrar_ruidos_float(df_test_copy, variaveis_float[indice])


Coluna: Annual_Income
Valores distintos com ruído: 3168
Total de registros com ruído: 3520

Ocorrências:
Annual_Income
36346.13_     3
101912.13_    3
16014.16_     3
97174.44_     3
97420.86_     3
             ..
35793.97_     1
18940.82_     1
14937.49_     1
22620.79_     1
38321.39_     1
Name: count, Length: 3168, dtype: int64

Coluna: Changed_Credit_Limit
Valores distintos com ruído: 1
Total de registros com ruído: 1059

Ocorrências:
Changed_Credit_Limit
_    1059
Name: count, dtype: int64

Coluna: Outstanding_Debt
Valores distintos com ruído: 482
Total de registros com ruído: 491

Ocorrências:
Outstanding_Debt
2593.23_    2
2266.48_    2
646.98_     2
644.34_     2
957.94_     2
           ..
100.3_      1
729.69_     1
559.18_     1
3711.23_    1
732.11_     1
Name: count, Length: 482, dtype: int64

Coluna: Amount_invested_monthly
Valores distintos com ruído: 1
Total de registros com ruído: 2175

Ocorrências:
Amount_invested_monthly
__10000__    2175
Name: count, dtype: int64

In [21]:
# limpando os valores com ruído
for indice, variaveis in enumerate(variaveis_float):
    limpeza_float.limpar_coluna_float(df_test_copy, variaveis)


Coluna: Annual_Income
Valores com '_' corrigidos: 3520
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0

Coluna: Changed_Credit_Limit
Valores com '_' corrigidos: 1059
Valores compostos apenas por '_': 1059
Valores convertidos para NaN: 1059

Coluna: Outstanding_Debt
Valores com '_' corrigidos: 491
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0

Coluna: Amount_invested_monthly
Valores com '_' corrigidos: 2175
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0

Coluna: Monthly_Balance
Valores com '_' corrigidos: 6
Valores compostos apenas por '_': 0
Valores convertidos para NaN: 0


In [22]:
# verificando se ainda restam ruídos
for indice, variaveis in enumerate(variaveis_float):
    limpeza_float.encontrar_ruidos_float(df_test_copy, variaveis_float[indice])


Coluna: Annual_Income
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Changed_Credit_Limit
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Outstanding_Debt
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Amount_invested_monthly
Valores distintos com ruído: 0
Total de registros com ruído: 0

Coluna: Monthly_Balance
Valores distintos com ruído: 1
Total de registros com ruído: 6

Ocorrências:
Monthly_Balance
-3.333333333333333e+26    6
Name: count, dtype: int64


In [23]:
# verificando os tipos de dados das colunas após a limpeza
for variaveis in variaveis_float:
    print(f'{variaveis}: {df_test_copy[variaveis].dtype}')

Annual_Income: float64
Changed_Credit_Limit: float64
Outstanding_Debt: float64
Amount_invested_monthly: float64
Monthly_Balance: float64


## 3. Tratando valores contínuos que deveriam ser discretos

In [24]:
problemas

,Problema identificado,Variáveis,Ação prevista
0,Valores numéricos em formato de string,"Age, Annual_Income, Num_of_Loan, Num_of_Delayed_Payment, Changed_Credit_Limit, Outstanding_Debt, Amount_invested_monthly, Monthly_Balance",Corrigir valores no formato incorreto e converter para número
1,Valores discretos em formato contínuo,Num_Credit_Inquiries,Converter para int
2,Variáveis que devem ter seus valores transformados,"Credit_History_Age, Payment_of_Min_Amount",Ver qual a melhor abordagem para cada caso
3,Valores inconsistentes com o domínio do problema,"Num_Bank_Accounts, Delay_from_due_date","Investigar e corrigir, ou substituir por valores ausentes"
4,Ruídos em variáveis categóricas,Verificar todas as variáveis,Investigar e substituir por valores ausentes
5,Variável alvo com instâncias desbalanceadas,Credit_Score,"Verificar necessidade de técnicas para balanceamento durante a modelagem, ou validação cruzada"
6,Variáveis identificadoras para serem excluídas,"ID, Customer_ID, Name, SSN",Excluir colunas na etapa de limpeza
7,Valores ausentes,"Monthly_Inhand_Salary, Type_of_Loan, Name, Credit_History_Age, Num_of_Delayed_Payment, Amount_invested_monthly, Num_Credit_Inquiries, Monthly_Balance",Definir estratégia de imputação


> Iremos converter os valores da variável `Num_Credit_Inquiries` para o tipo int.

In [25]:
coluna = 'Num_Credit_Inquiries'
print(f'Tipo de dado da coluna {coluna}: {df_train_copy[coluna].dtype}')

Tipo de dado da coluna Num_Credit_Inquiries: float64


In [26]:
# convertendo os valores para int
df_train_copy[coluna] = df_train_copy[coluna].astype('Int64')

print(f'Tipo de dado da coluna {coluna}: {df_train_copy[coluna].dtype}')
df_train_copy['Num_Credit_Inquiries']

Tipo de dado da coluna Num_Credit_Inquiries: Int64


0        4
1        4
2        4
3        4
4        4
        ..
99995    3
99996    3
99997    3
99998    3
99999    3
Name: Num_Credit_Inquiries, Length: 100000, dtype: Int64

## 4. Tratando variáveis que precisam ter seus valores transformados

Na exploração incial, foi identificada que duas variáveis deveriam ter seus valores transformados para trazerem informações mais significativas:

- `Payment_of_Min_Amount` é uma variável binária e aparece como "Sim" ou "Não". Podemos converter esses valores para 0 (Não) e 1 (Sim).
- `Credit_History_Age` aparece como um valor textual seguindo o padrão "x Years and y Months". Porém, é possível representá-la como um valor numérico, então podemos converter o valor para a quantidade total de meses e mudar o nome da coluna para `Credit_History_Months`.

### 4.1 Tratando `Payment_of_Min_Amount`

#### 4.1.1 Tratando no dataset de treino

In [27]:
df_train_copy['Payment_of_Min_Amount'].value_counts()

Payment_of_Min_Amount
Yes    52326
No     35667
NM     12007
Name: count, dtype: int64

> "Yes" será convertido para 1, "No" será convertido para 0 e "NM" virará dado nulo.

In [28]:
coluna = 'Payment_of_Min_Amount'

df_train_copy[coluna] = (
    df_train_copy[coluna]
    .replace({
        "Yes": 1,
        "No": 0,
        "NM": np.nan
    })
    .astype("Int64")
)

df_train_copy['Payment_of_Min_Amount'].value_counts()

Payment_of_Min_Amount
1    52326
0    35667
Name: count, dtype: Int64

#### 4.1.2 Tratando no dataset de teste

In [29]:
# tratando no dataset de teste
df_test_copy[coluna] = (
    df_test_copy[coluna]
    .replace({
        "Yes": 1,
        "No": 0,
        "NM": np.nan
    })
    .astype("Int64")
)

df_test_copy['Payment_of_Min_Amount'].value_counts()

Payment_of_Min_Amount
1    26158
0    17849
Name: count, dtype: Int64

### 4.2 Tratando `Credit_History_Age`

#### 4.2.1 Tratando no dataset de treino

In [30]:
df_train_copy['Credit_History_Age'].value_counts()

Credit_History_Age
15 Years and 11 Months    446
19 Years and 4 Months     445
19 Years and 5 Months     444
17 Years and 11 Months    443
19 Years and 3 Months     441
                         ... 
0 Years and 3 Months       20
0 Years and 2 Months       15
33 Years and 7 Months      14
33 Years and 8 Months      12
0 Years and 1 Months        2
Name: count, Length: 404, dtype: int64

In [31]:
print(f'Quantidade de dados nulos em Credit_History_Age: {df_train_copy['Credit_History_Age'].isnull().sum()}')

Quantidade de dados nulos em Credit_History_Age: 9030


>Vamos extrair a quantidade de anos, convertê-los para meses e somá-lo com os meses que são informados também. Antes da conversão, a quantidade de dados nulos presentes nessa coluna é **9030**.

In [32]:
from src.funcoes_limpeza.limpar_credit_history_age import limpar_credit_history_age

limpar_credit_history_age(df_train_copy)

Coluna: Credit_History_Age
Valores convertidos: 90970
Valores ausentes: 9030


In [33]:
df_train_copy['Credit_History_Age'].value_counts()

Credit_History_Age
191    446
232    445
233    444
215    443
231    441
      ... 
3       20
2       15
403     14
404     12
1        2
Name: count, Length: 404, dtype: Int64

> Os valores de `Credit_History_Age` foram convertidos para números e nenhum novo valor ausente foi criado.

In [34]:
# mudando a informação sobre a variável no dicionario
mascara = dicionario["Variável"] == "Credit_History_Age"

dicionario.loc[mascara, ["Variável", "Descrição"]] = [
    "Credit_History_Months",
    "Quantidade total de meses do histórico de crédito do cliente"
]

display(dicionario.loc[dicionario["Variável"] == "Credit_History_Months"])

# mudando o nome da coluna no dataframe de treino
df_train_copy.rename(columns={'Credit_History_Age': 'Credit_History_Months'}, inplace=True)

,Variável,Tipo,Subtipo,Papel,Descrição
21,Credit_History_Months,numérica,discreta,preditora,Quantidade total de meses do histórico de crédito do cliente


#### 4.2.2 Tratando no dataset de teste

In [35]:
# limpando no dataset de teste
limpar_credit_history_age(df_test_copy)

Coluna: Credit_History_Age
Valores convertidos: 45530
Valores ausentes: 4470


In [36]:
# mudando o nome da coluna no dataframe de teste
df_test_copy.rename(columns={'Credit_History_Age': 'Credit_History_Months'}, inplace=True)

## 5. Tratando valores inconsistentes com o domínio do problema

In [37]:
problemas

,Problema identificado,Variáveis,Ação prevista
0,Valores numéricos em formato de string,"Age, Annual_Income, Num_of_Loan, Num_of_Delayed_Payment, Changed_Credit_Limit, Outstanding_Debt, Amount_invested_monthly, Monthly_Balance",Corrigir valores no formato incorreto e converter para número
1,Valores discretos em formato contínuo,Num_Credit_Inquiries,Converter para int
2,Variáveis que devem ter seus valores transformados,"Credit_History_Age, Payment_of_Min_Amount",Ver qual a melhor abordagem para cada caso
3,Valores inconsistentes com o domínio do problema,"Num_Bank_Accounts, Delay_from_due_date","Investigar e corrigir, ou substituir por valores ausentes"
4,Ruídos em variáveis categóricas,Verificar todas as variáveis,Investigar e substituir por valores ausentes
5,Variável alvo com instâncias desbalanceadas,Credit_Score,"Verificar necessidade de técnicas para balanceamento durante a modelagem, ou validação cruzada"
6,Variáveis identificadoras para serem excluídas,"ID, Customer_ID, Name, SSN",Excluir colunas na etapa de limpeza
7,Valores ausentes,"Monthly_Inhand_Salary, Type_of_Loan, Name, Credit_History_Age, Num_of_Delayed_Payment, Amount_invested_monthly, Num_Credit_Inquiries, Monthly_Balance",Definir estratégia de imputação


- `Num_Bank_Accounts` apresenta valores negativos, e isso não faz sentido para a quantidade de contas bancárias de um cliente.
- `Delay_from_due_date` apresenta valores negativos, o que não faz sentido para a quantidade de dias de atraso de um pagamento.

In [38]:
# procurando os valores negativos
colunas = ['Num_Bank_Accounts', 'Delay_from_due_date']
for coluna in colunas:
    limpeza_int.encontrar_ruidos_int(df_train_copy, coluna)


Coluna: Num_Bank_Accounts
Valores distintos com ruído: 1
Total de registros com ruído: 21

Ocorrências:
Num_Bank_Accounts
-1    21
Name: count, dtype: int64

Coluna: Delay_from_due_date
Valores distintos com ruído: 5
Total de registros com ruído: 591

Ocorrências:
Delay_from_due_date
-1    210
-2    168
-3    118
-4     62
-5     33
Name: count, dtype: int64


In [39]:
# limpando do dataset de treino
for coluna in colunas:
    limpeza_int.remover_negativos_int(df_train_copy, coluna)


Coluna: Num_Bank_Accounts
Valores negativos convertidos para NaN: 21

Coluna: Delay_from_due_date
Valores negativos convertidos para NaN: 591


In [40]:
# procurando os valores negativos
for coluna in colunas:
    limpeza_int.encontrar_ruidos_int(df_test_copy, coluna)


Coluna: Num_Bank_Accounts
Valores distintos com ruído: 1
Total de registros com ruído: 16

Ocorrências:
Num_Bank_Accounts
-1    16
Name: count, dtype: int64

Coluna: Delay_from_due_date
Valores distintos com ruído: 5
Total de registros com ruído: 298

Ocorrências:
Delay_from_due_date
-1    101
-2     71
-3     59
-4     49
-5     18
Name: count, dtype: int64


In [41]:
# limpando do dataset de teste
for coluna in colunas:
    limpeza_int.remover_negativos_int(df_test_copy, coluna)


Coluna: Num_Bank_Accounts
Valores negativos convertidos para NaN: 16

Coluna: Delay_from_due_date
Valores negativos convertidos para NaN: 298


---
Os valores negativos foram removidos das colunas `Num_Bank_Accounts` e `Delay_from_due_date`, agora só falta tratar os valores ausentes e possíveis outliers extremos para essas colunas.

---

## 6. Verificando ruídos em variáveis categóricas

In [42]:
variaveis_categoricas = (
    dicionario.loc[
        (dicionario["Tipo"] == "categórica") & (dicionario["Papel"] == "preditora"),
        "Variável"
    ]
    .tolist()
)

print(variaveis_categoricas)

['Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']


In [43]:
for variavel in variaveis_categoricas:
    display(df_train_copy[variavel].value_counts())
    print('\n')

Month
January     12500
February    12500
March       12500
April       12500
May         12500
June        12500
July        12500
August      12500
Name: count, dtype: int64

Occupation
_______          7062
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64

Type_of_Loan
Not Specified                                                                                        1408
Credit-Builder Loan                                                                                  1280
Personal Loan                                                                                        1272
Debt Consolidation Loan                                                                              1264
Student Loan                                                                                         1240
                                                                                                     ... 
Auto Loan, Payday Loan, Auto Loan, Student Loan, Student Loan, Student Loan, and Home Equity Loan       8
Home Equity Loan, Payday Loan, Not Specified, and Personal Loan                                         8
Home Equity Loan, Auto Loan, Auto Loan, and Auto Loan                                                   8
Payday Loan, Student Loan, Mortga

Credit_Mix
Standard    36479
Good        24337
_           20195
Bad         18989
Name: count, dtype: int64

Payment_of_Min_Amount
1    52326
0    35667
Name: count, dtype: Int64

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
!@9#%8                               7600
Name: count, dtype: int64

Verificamos os seguintes ruídos:

- `Occupation` possui o valor _______ que deve ser convertido para nulo
- `Type_of_Loan` tem o valor Not Specified, que deve virar nulo também
- `Credit_Mix` tem  _ que também é nulo
- `Payment_Behaviour` possui o ruído !@9#%8, que deve virar nulo também

In [44]:
# transformando os valores para nulos no dataset de treino
df_train_copy['Occupation'] = df_train_copy['Occupation'].replace('_______', np.nan)
df_train_copy['Type_of_Loan'] = df_train_copy['Type_of_Loan'].replace('Not Specified', np.nan)
df_train_copy['Credit_Mix'] = df_train_copy['Credit_Mix'].replace('_', np.nan)
df_train_copy['Payment_Behaviour'] = df_train_copy['Payment_Behaviour'].replace('!@9#%8', np.nan)

In [45]:
for variavel in variaveis_categoricas:
    display(df_train_copy[variavel].value_counts())
    print('\n')

Month
January     12500
February    12500
March       12500
April       12500
May         12500
June        12500
July        12500
August      12500
Name: count, dtype: int64

Occupation
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64

Type_of_Loan
Credit-Builder Loan                                                                                  1280
Personal Loan                                                                                        1272
Debt Consolidation Loan                                                                              1264
Student Loan                                                                                         1240
Payday Loan                                                                                          1200
                                                                                                     ... 
Auto Loan, Payday Loan, Auto Loan, Student Loan, Student Loan, Student Loan, and Home Equity Loan       8
Home Equity Loan, Payday Loan, Not Specified, and Personal Loan                                         8
Home Equity Loan, Auto Loan, Auto Loan, and Auto Loan                                                   8
Payday Loan, Student Loan, Mortga

Credit_Mix
Standard    36479
Good        24337
Bad         18989
Name: count, dtype: int64

Payment_of_Min_Amount
1    52326
0    35667
Name: count, dtype: Int64

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
Name: count, dtype: int64

In [46]:
# verificando o dataset de teste
for variavel in variaveis_categoricas:
    display(df_test_copy[variavel].value_counts())
    print('\n')

Month
September    12500
October      12500
November     12500
December     12500
Name: count, dtype: int64

Occupation
_______          3438
Lawyer           3324
Engineer         3212
Architect        3195
Mechanic         3168
Developer        3146
Accountant       3133
Media_Manager    3130
Scientist        3104
Teacher          3103
Entrepreneur     3103
Journalist       3037
Doctor           3027
Manager          3000
Musician         2947
Writer           2933
Name: count, dtype: int64

Type_of_Loan
Not Specified                                                                                        704
Credit-Builder Loan                                                                                  640
Personal Loan                                                                                        636
Debt Consolidation Loan                                                                              632
Student Loan                                                                                         620
                                                                                                    ... 
Auto Loan, Payday Loan, Auto Loan, Student Loan, Student Loan, Student Loan, and Home Equity Loan      4
Home Equity Loan, Payday Loan, Not Specified, and Personal Loan                                        4
Home Equity Loan, Auto Loan, Auto Loan, and Auto Loan                                                  4
Payday Loan, Student Loan, Mortgage Loan, 

Credit_Mix
Standard    18379
Good        12260
_            9805
Bad          9556
Name: count, dtype: int64

Payment_of_Min_Amount
1    26158
0    17849
Name: count, dtype: Int64

Payment_Behaviour
Low_spent_Small_value_payments      12694
High_spent_Medium_value_payments     8922
High_spent_Large_value_payments      6844
Low_spent_Medium_value_payments      6837
High_spent_Small_value_payments      5651
Low_spent_Large_value_payments       5252
!@9#%8                               3800
Name: count, dtype: int64

In [47]:
# transformando os valores para nulos no dataset de teste
df_test_copy['Occupation'] = df_test_copy['Occupation'].replace('_______', np.nan)
df_test_copy['Type_of_Loan'] = df_test_copy['Type_of_Loan'].replace('Not Specified', np.nan)
df_test_copy['Credit_Mix'] = df_test_copy['Credit_Mix'].replace('_', np.nan)
df_test_copy['Payment_Behaviour'] = df_test_copy['Payment_Behaviour'].replace('!@9#%8', np.nan)

## 7. Tratando outliers extremos de dados numéricos

### 7.1 Tratando no dataset de treino

In [48]:
df_train_copy.describe()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Months,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
count,99114.0,1.000000e+05,84998.000000,99979.0,100000.00000,100000.000000,96124.0,99409.0,92354.0,97909.000000,98035.0,100000.000000,100000.000000,90970.0,87993.0,100000.000000,95521.000000,9.880000e+04
mean,116.108421,1.764157e+05,4194.170850,17.09508,22.47443,72.466040,7.163622,21.207245,31.150519,10.389025,27.754251,1426.220376,32.285173,221.195405,0.594661,1403.118217,637.412998,-3.036437e+22
std,686.861274,1.429618e+06,3183.686167,117.416871,129.05741,466.422621,60.314923,14.794693,226.802173,6.789496,193.177339,1155.129026,5.116875,99.741364,0.49096,8306.041270,2043.319327,3.181295e+24
min,14.0,7.005930e+03,303.645417,0.0,0.00000,1.000000,0.0,0.0,0.0,-6.490000,0.0,0.230000,20.000000,1.0,0.0,0.000000,0.000000,-3.333333e+26
25%,25.0,1.945750e+04,1625.568229,3.0,4.00000,8.000000,2.0,10.0,9.0,5.320000,3.0,566.072500,28.052567,144.0,0.0,30.306660,74.534002,2.700922e+02
50%,33.0,3.757861e+04,3093.745000,6.0,5.00000,13.000000,3.0,18.0,14.0,9.400000,6.0,1166.155000,32.305784,219.0,1.0,69.249473,135.925682,3.367192e+02
75%,42.0,7.279092e+04,5957.448333,7.0,7.00000,20.000000,5.0,28.0,18.0,14.870000,9.0,1945.962500,36.496663,302.0,1.0,161.224249,265.731733,4.702202e+02
max,8698.0,2.419806e+07,15204.633333,1798.0,1499.00000,5797.000000,1496.0,67.0,4397.0,36.970000,2597.0,4998.070000,50.000000,404.0,1.0,82331.000000,10000.000000,1.602041e+03


- `Age` possui seu valor máximo como 8698, o que é uma idade impossível. Iremos estabelecer o limite máximo de idade para **120** anos e remover os valores que são maiores que isso
- `Num_Bank_Accounts` tem valor máximo de 1798, iremos estabelecer o limite máximo para **20** (é muito incomum uma pessoa física possuir mais de 20 contas bancárias)
- `Num_Credit_Card` tem valor máximo de 1499, iremos estabelecer o limite máximo para **20** (mesmo clientes de alta renda dificilmente possuem mais de 20 cartões)
- `Interest_Rate` tem valor máximo de 5797, iremos estabelecer o limite máximo para **100** (mesmo empréstimos muito caros raramente ultrapassam algumas centenas de porcento ao ano)
- `Num_of_Loan` tem valor máximo de 1496, iremos estabelecer o limite máximo para **20** (pessoas podem possuir vários empréstimos, mas centenas são impossíveis)
- `Num_of_Delayed_Payment` tem valor máximo de 4397, iremos estabelecer o limite máximo para **100** (O número representa pagamentos atrasados ao longo do histórico. Valores na casa dos milhares são impossíveis, e um limite de 100 é bastante conservador)
- `Num_Credit_Inquiries` tem valor máximo de 2597, iremos estabelecer o limite máximo para **50** (consultas ao crédito acima de 50 já são extremamente incomuns e geralmente indicam erro)
- `Monthly_Balance` tem valor mínimo de -3.333333e+26. Iremos substituí-lo por dados nulos

In [49]:
df_train_copy.loc[df_train_copy['Age'] > 120, 'Age'] = np.nan
df_train_copy.loc[df_train_copy['Num_Bank_Accounts'] > 20, 'Num_Bank_Accounts'] = np.nan
df_train_copy.loc[df_train_copy['Num_Credit_Card'] > 20, 'Num_Credit_Card'] = np.nan
df_train_copy.loc[df_train_copy['Interest_Rate'] > 100, 'Interest_Rate'] = np.nan
df_train_copy.loc[df_train_copy['Num_of_Loan'] > 20, 'Num_of_Loan'] = np.nan
df_train_copy.loc[df_train_copy['Num_of_Delayed_Payment'] > 100, 'Num_of_Delayed_Payment'] = np.nan
df_train_copy.loc[df_train_copy['Num_Credit_Inquiries'] > 50, 'Num_Credit_Inquiries'] = np.nan
df_train_copy['Monthly_Balance'] = df_train_copy['Monthly_Balance'].replace(-333333333333333333333333333, np.nan)

In [50]:
df_train_copy.describe()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Months,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
count,97230.0,1.000000e+05,84998.000000,98665.0,97737.000000,97988.000000,95655.0,99409.0,91630.0,97909.000000,96401.0,100000.000000,100000.000000,90970.0,87993.0,100000.000000,95521.000000,98791.000000
mean,33.328078,1.764157e+05,4194.170850,5.369108,5.534219,14.546679,3.534211,21.207245,13.427491,10.389025,5.786133,1426.220376,32.285173,221.195405,0.594661,1403.118217,637.412998,402.551258
std,10.79598,1.429618e+06,3183.686167,2.592749,2.070051,8.798523,2.448614,14.794693,6.248467,6.789496,3.881031,1155.129026,5.116875,99.741364,0.49096,8306.041270,2043.319327,213.925499
min,14.0,7.005930e+03,303.645417,0.0,0.000000,1.000000,0.0,0.0,0.0,-6.490000,0.0,0.230000,20.000000,1.0,0.0,0.000000,0.000000,0.007760
25%,24.0,1.945750e+04,1625.568229,3.0,4.000000,7.000000,2.0,10.0,9.0,5.320000,3.0,566.072500,28.052567,144.0,0.0,30.306660,74.534002,270.106630
50%,33.0,3.757861e+04,3093.745000,5.0,5.000000,13.000000,3.0,18.0,14.0,9.400000,5.0,1166.155000,32.305784,219.0,1.0,69.249473,135.925682,336.731225
75%,42.0,7.279092e+04,5957.448333,7.0,7.000000,20.000000,5.0,28.0,18.0,14.870000,8.0,1945.962500,36.496663,302.0,1.0,161.224249,265.731733,470.262938
max,118.0,2.419806e+07,15204.633333,18.0,20.000000,100.000000,19.0,67.0,98.0,36.970000,49.0,4998.070000,50.000000,404.0,1.0,82331.000000,10000.000000,1602.040519


### 7.2 Tratando no dataset de teste

In [51]:
df_test_copy.describe()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Months,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
count,49536.0,5.000000e+04,42502.000000,49984.0,50000.000000,50000.000000,48026.0,49702.0,46215.0,48941.000000,48965.000000,50000.000000,50000.000000,45530.0,44007.0,50000.000000,47729.000000,4.943800e+04
mean,115.425569,1.663342e+05,4182.004291,16.84397,22.921480,68.772640,7.653521,21.193071,31.09155,10.374844,30.080200,1426.220376,32.279581,227.251175,0.594405,1491.304305,641.654795,-4.045471e+22
std,680.293906,1.351965e+06,3174.109304,116.415039,129.314804,451.602363,64.246844,14.793139,222.182238,6.780628,196.984121,1155.134801,5.106238,99.554366,0.491012,8595.647887,2053.895420,3.671994e+24
min,14.0,7.005930e+03,303.645417,0.0,0.000000,1.000000,0.0,0.0,0.0,-6.450000,0.000000,0.230000,20.509652,10.0,0.0,0.000000,0.000000,-3.333333e+26
25%,25.0,1.945333e+04,1625.188333,3.0,4.000000,8.000000,2.0,10.0,9.0,5.340000,4.000000,566.072500,28.061040,151.0,0.0,32.222388,74.529270,2.704741e+02
50%,34.0,3.757782e+04,3086.305000,6.0,5.000000,13.000000,3.0,18.0,14.0,9.410000,7.000000,1166.155000,32.280390,225.0,1.0,74.733349,135.590430,3.369732e+02
75%,43.0,7.281702e+04,5934.189094,7.0,7.000000,20.000000,5.0,28.0,18.0,14.800000,10.000000,1945.962500,36.468591,307.0,1.0,176.157491,266.892228,4.708568e+02
max,8688.0,2.413726e+07,15204.633333,1798.0,1499.000000,5799.000000,1496.0,67.0,4399.0,36.650000,2593.000000,4998.070000,48.540663,408.0,1.0,82398.000000,10000.000000,1.606518e+03


In [52]:
df_test_copy.loc[df_test_copy['Age'] > 120, 'Age'] = np.nan
df_test_copy.loc[df_test_copy['Num_Bank_Accounts'] > 20, 'Num_Bank_Accounts'] = np.nan
df_test_copy.loc[df_test_copy['Num_Credit_Card'] > 20, 'Num_Credit_Card'] = np.nan
df_test_copy.loc[df_test_copy['Interest_Rate'] > 100, 'Interest_Rate'] = np.nan
df_test_copy.loc[df_test_copy['Num_of_Loan'] > 20, 'Num_of_Loan'] = np.nan
df_test_copy.loc[df_test_copy['Num_of_Delayed_Payment'] > 100, 'Num_of_Delayed_Payment'] = np.nan
df_test_copy.loc[df_test_copy['Num_Credit_Inquiries'] > 50, 'Num_Credit_Inquiries'] = np.nan
df_test_copy['Monthly_Balance'] = df_test_copy['Monthly_Balance'].replace(-333333333333333333333333333, np.nan)

## 8. Tratando valores ausentes

Após os tratamentos feitos nas seções anteriores, novos dados ausentes foram criados, então iremos incluí-los nesta etapa de limpeza também.

In [53]:
from src.funcoes_auxiliares.resumo_nulos import resumo_nulos
resumo_nulos(df_train_copy, "TREINO")

Valores nulos para o dataset de TREINO:


,Quantidade de nulos,Porcentagem (%)
Credit_Mix,20195,20.20
Monthly_Inhand_Salary,15002,15.00
Type_of_Loan,12816,12.82
Payment_of_Min_Amount,12007,12.01
Name,9985,9.98
Credit_History_Months,9030,9.03
Num_of_Delayed_Payment,8370,8.37
Payment_Behaviour,7600,7.60
Occupation,7062,7.06
Amount_invested_monthly,4479,4.48


### 8.1 Tratando variáveis que podem ser imputadas considerando o `Custumer_ID`

Algumas variáveis devem ter somente um valor por cliente. Para elas, vamos imputar valores ausentes considerando os valores de outras células que possuam o mesmo `Custumer_ID`. Faremos isso para as variáveis a seguir:
- Occupation
- Monthly_Inhand_Salary

Mas antes de imputar, precisamos verificar se realmente há apenas um único valor por cliente nessas colunas.

#### 8.1.1 Tratando no dataset de treino

In [54]:
import src.funcoes_auxiliares.tratar_valores_por_id as valores_por_id

# verificando a quantidade de valores distintos que aparecem nessas colunas por cada id
colunas = ['Occupation', 'Monthly_Inhand_Salary']
for coluna in colunas:
    valores_por_id.verificar_constancia(df_train_copy, coluna)


Constância da coluna: Occupation
   Quantidade de valores distintos  Quantidade de Customer_IDs
0                                1                       12500

Constância da coluna: Monthly_Inhand_Salary
   Quantidade de valores distintos  Quantidade de Customer_IDs
0                                1                       11754
1                                2                         746


---
Se um Customer_ID possui mais de um valor, isso provavelmente é um erro de qualidade dos dados. Vemos que não há apenas um valor único para as colunas especificadas. Então para a imputação iremos substituir todos os valores pelo valor mais frequente (moda) daquele cliente.

---

In [55]:
# corrigindo colunas que possuem mais de um valor por Custumer_ID
for coluna in colunas:
    valores_por_id.corrigir_por_moda_customer_id(df_train_copy, coluna, id_col="Customer_ID")


Coluna: Occupation
Customer_IDs corrigidos: 0

Coluna: Monthly_Inhand_Salary
Customer_IDs corrigidos: 746


In [56]:
for coluna in colunas:
    valores_por_id.verificar_constancia(df_train_copy, coluna)


Constância da coluna: Occupation
   Quantidade de valores distintos  Quantidade de Customer_IDs
0                                1                       12500

Constância da coluna: Monthly_Inhand_Salary
   Quantidade de valores distintos  Quantidade de Customer_IDs
0                                1                       12500


In [57]:
# imputando os valores por Custumer_ID
df_train_copy = valores_por_id.imputar_nulos_por_moda_customer_id(df_train_copy, colunas, id_col="Customer_ID")
resumo_nulos(df_train_copy, "TREINO")

Valores nulos para o dataset de TREINO:


,Quantidade de nulos,Porcentagem (%)
Credit_Mix,20195,20.20
Type_of_Loan,12816,12.82
Payment_of_Min_Amount,12007,12.01
Name,9985,9.98
Credit_History_Months,9030,9.03
Num_of_Delayed_Payment,8370,8.37
Payment_Behaviour,7600,7.60
Amount_invested_monthly,4479,4.48
Num_of_Loan,4345,4.35
Num_Credit_Inquiries,3599,3.60


#### 8.1.2 Tratando no dataset de teste

In [58]:
# verificando a quantidade de valores distintos que aparecem nessas colunas por cada id
for coluna in colunas:
    valores_por_id.verificar_constancia(df_test_copy, coluna)


Constância da coluna: Occupation
   Quantidade de valores distintos  Quantidade de Customer_IDs
0                                1                       12500

Constância da coluna: Monthly_Inhand_Salary
   Quantidade de valores distintos  Quantidade de Customer_IDs
0                                0                           7
1                                1                       12183
2                                2                         310


In [59]:
# corrigindo colunas que possuem mais de um valor por Custumer_ID
colunas = ['Occupation', 'Monthly_Inhand_Salary']
for coluna in colunas:
    valores_por_id.corrigir_por_moda_customer_id(df_test_copy, coluna, id_col="Customer_ID")


Coluna: Occupation
Customer_IDs corrigidos: 0

Coluna: Monthly_Inhand_Salary
Customer_IDs corrigidos: 310


In [60]:
# imputando os valores por Custumer_ID
df_test_copy = valores_por_id.imputar_nulos_por_moda_customer_id(df_test_copy, colunas, id_col="Customer_ID")
resumo_nulos(df_test_copy, "TESTE")

Valores nulos para o dataset de TESTE:


,Quantidade de nulos,Porcentagem (%)
Credit_Mix,9805,19.61
Type_of_Loan,6408,12.82
Payment_of_Min_Amount,5993,11.99
Name,5015,10.03
Credit_History_Months,4470,8.94
Num_of_Delayed_Payment,4175,8.35
Payment_Behaviour,3800,7.60
Amount_invested_monthly,2271,4.54
Num_of_Loan,2232,4.46
Num_Credit_Inquiries,1876,3.75


---
Todos os valores nulos das variáveis `Occupation` e `Monthly_Inhand_Salary` foram preenchidos no dataset de treino. No dataset de teste ainda restaram alguns valores nulos, mas eles serão imputados posteriormente por mediana. A seguir, preencheremos as variáveis categóricas com a moda do dataset e os valores numéricos com a mediana.

---

### 8.2 Imputando variáveis categóricas pela moda

In [61]:
variaveis_categoricas = (
    dicionario
    .query("Tipo == 'categórica' and Papel == 'preditora'")
    ["Variável"]
    .tolist()
)
print(variaveis_categoricas)

['Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']


In [62]:
# preenchendo com a moda para o dataset de treino
for variavel in variaveis_categoricas:
    moda = df_train_copy[variavel].mode(dropna=True)

    if not moda.empty:
        df_train_copy[variavel] = df_train_copy[variavel].fillna(moda.iloc[0])

resumo_nulos(df_train_copy, "TREINO")

Valores nulos para o dataset de TREINO:


,Quantidade de nulos,Porcentagem (%)
Name,9985,9.98
Credit_History_Months,9030,9.03
Num_of_Delayed_Payment,8370,8.37
Amount_invested_monthly,4479,4.48
Num_of_Loan,4345,4.35
Num_Credit_Inquiries,3599,3.60
Age,2770,2.77
Num_Credit_Card,2263,2.26
Changed_Credit_Limit,2091,2.09
Interest_Rate,2012,2.01


In [63]:
# preenchendo com a moda para o dataset de teste
for variavel in variaveis_categoricas:
    moda = df_test_copy[variavel].mode(dropna=True)

    if not moda.empty:
        df_test_copy[variavel] = df_test_copy[variavel].fillna(moda.iloc[0])

resumo_nulos(df_train_copy, "TESTE")

Valores nulos para o dataset de TESTE:


,Quantidade de nulos,Porcentagem (%)
Name,9985,9.98
Credit_History_Months,9030,9.03
Num_of_Delayed_Payment,8370,8.37
Amount_invested_monthly,4479,4.48
Num_of_Loan,4345,4.35
Num_Credit_Inquiries,3599,3.60
Age,2770,2.77
Num_Credit_Card,2263,2.26
Changed_Credit_Limit,2091,2.09
Interest_Rate,2012,2.01


### 8.3 Imputando variáveis numéricas pela mediana

In [64]:
variaveis_numericas = (
    dicionario
    .query("Tipo == 'numérica' and Papel == 'preditora'")
    ["Variável"]
    .tolist()
)
print(variaveis_numericas)

['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Months', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']


In [65]:
# preenchendo com a mediana para o dataset de treino
for variavel in variaveis_numericas:
    mediana = df_train_copy[variavel].median(skipna=True)
    df_train_copy[variavel] = df_train_copy[variavel].fillna(mediana)

resumo_nulos(df_train_copy, "TREINO")

Valores nulos para o dataset de TREINO:


,Quantidade de nulos,Porcentagem (%)
Name,9985,9.98


In [66]:
# preenchendo com a mediana para o dataset de teste
for variavel in variaveis_numericas:
    mediana = df_test_copy[variavel].median(skipna=True)
    df_test_copy[variavel] = df_test_copy[variavel].fillna(mediana)

resumo_nulos(df_test_copy, "TREINO")

Valores nulos para o dataset de TREINO:


,Quantidade de nulos,Porcentagem (%)
Name,5015,10.03


## 9. Excluindo colunas identificadoras

As variáveis do tipo identificadoras não possuem nenhum poder preditivo para o problema de classificação de score de crédito, então excluiremos essas colunas para otimizar os dados para o modelo.

In [67]:
colunas_identificadoras = (
    dicionario
    .query("Tipo == 'categórica' and Papel == 'identificador'")
    ["Variável"]
    .tolist()
)

print(colunas_identificadoras)

['ID', 'Customer_ID', 'Name', 'SSN']


In [68]:
# removendo colunas no dataset de treino
for coluna in colunas_identificadoras:
    if coluna in df_train_copy.columns:
        df_train_copy.drop(columns=coluna, inplace=True)

df_train_copy.columns

Index(['Month', 'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
       'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Months',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='str')

In [69]:
# removendo colunas no dataset de teste
for coluna in colunas_identificadoras:
    if coluna in df_test_copy.columns:
        df_test_copy.drop(columns=coluna, inplace=True)

df_test_copy.columns

Index(['Month', 'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
       'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Months',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance'],
      dtype='str')

## 10. Conclusão

As variáveis foram convertidas para os tipos que devem ser e não há mais dados nulos nos conjuntos de dados de teste e de treino. Podemos passar para a análise exploratória de dados, para posteriormente fazer o pré-processamento de dados e a construção dos modelos de classificação.

In [70]:
from src.funcoes_auxiliares.comparar_tipos_dados import comparar_tipos_dados

comparacao = comparar_tipos_dados(df_train_copy, df_test_copy)
comparacao

,Tipo_train,Tipo_test,Mesmo_tipo
Month,str,str,True
Age,Int64,Int64,True
Occupation,str,str,True
Annual_Income,float64,float64,True
Monthly_Inhand_Salary,float64,float64,True
Num_Bank_Accounts,Int64,Int64,True
Num_Credit_Card,float64,float64,True
Interest_Rate,float64,float64,True
Num_of_Loan,Int64,Int64,True
Type_of_Loan,str,str,True


In [71]:
resumo_nulos(df_train_copy, 'TREINO')

Valores nulos para o dataset de TREINO:


,Quantidade de nulos,Porcentagem (%)


In [72]:
resumo_nulos(df_test_copy, 'TESTE')

Valores nulos para o dataset de TESTE:


,Quantidade de nulos,Porcentagem (%)


In [73]:
# salvando os datasets limpos
df_train_copy.to_csv('../data/processed/train_clean.csv', index=False)
df_test_copy.to_csv('../data/processed/test_clean.csv', index=False)